In [4]:
import boto3
s3 = boto3.client("s3")
resp = s3.list_objects_v2(Bucket="patient-risk-mlops", Prefix="raw/")
for o in resp.get("Contents", []):
    print(o["Key"], o["Size"])

raw/ 0
raw/human_vital_signs_dataset_2024.csv 38793004


In [5]:
!pip install -q xgboost mlflow skl2onnx onnxruntime

In [6]:
import pandas as pd, numpy as np, re

df = pd.read_csv("s3://patient-risk-mlops/raw/human_vital_signs_dataset_2024.csv")
df.columns = [re.sub(r'[^a-z0-9]+','_',c.strip().lower()).strip('_') for c in df.columns]

print(df.shape)
print(df.columns.tolist())
print("Patient IDs unique:", df['patient_id'].is_unique)
print("Missing:", df.isnull().sum().sum(), " Duplicates:", df.duplicated().sum())
print(df['risk_category'].value_counts(normalize=True))
print(df.groupby('risk_category')[['heart_rate','oxygen_saturation','systolic_blood_pressure','derived_map','derived_bmi']].mean())

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:305: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


(200020, 17)
['patient_id', 'heart_rate', 'respiratory_rate', 'timestamp', 'body_temperature', 'oxygen_saturation', 'systolic_blood_pressure', 'diastolic_blood_pressure', 'age', 'gender', 'weight_kg', 'height_m', 'derived_hrv', 'derived_pulse_pressure', 'derived_bmi', 'derived_map', 'risk_category']
Patient IDs unique: True
Missing: 0  Duplicates: 0
risk_category
High Risk    0.525522
Low Risk     0.474478
Name: proportion, dtype: float64
               heart_rate  oxygen_saturation  systolic_blood_pressure  \
risk_category                                                           
High Risk       83.578795          97.503478               124.408905   
Low Risk        75.053527          97.505363               124.470165   

               derived_map  derived_bmi  
risk_category                            
High Risk        94.482538    25.933353  
Low Risk         94.475237    23.973876  


In [7]:
import warnings; warnings.filterwarnings("ignore")
num = df.select_dtypes('number').drop(columns=['patient_id'])
g = df.groupby('risk_category')[num.columns].mean().T
g['diff_%'] = (g['High Risk'] - g['Low Risk']) / g['Low Risk'] * 100
print(g.round(3))
print(pd.crosstab(df['gender'], df['risk_category'], normalize='index').round(3))

risk_category             High Risk  Low Risk  diff_%
heart_rate                   83.579    75.054  11.359
respiratory_rate             15.487    15.493  -0.039
body_temperature             36.749    36.748   0.003
oxygen_saturation            97.503    97.505  -0.002
systolic_blood_pressure     124.409   124.470  -0.049
diastolic_blood_pressure     79.519    79.478   0.052
age                          53.495    53.392   0.192
weight_kg                    75.335    74.621   0.957
height_m                      1.737     1.764  -1.501
derived_hrv                   0.100     0.100  -0.088
derived_pulse_pressure       44.890    44.992  -0.229
derived_bmi                  25.933    23.974   8.173
derived_map                  94.483    94.475   0.008
risk_category  High Risk  Low Risk
gender                            
Female             0.527     0.473
Male               0.524     0.476


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

feats = ['heart_rate','respiratory_rate','body_temperature','oxygen_saturation',
         'systolic_blood_pressure','diastolic_blood_pressure','age','weight_kg','height_m']
X = df[feats]
y = (df['risk_category'] == 'High Risk').astype(int)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42).fit(Xtr, ytr)
print("Accuracy:", round(accuracy_score(yte, rf.predict(Xte)), 4))
print("Majority-class baseline:", round(max(y.mean(), 1 - y.mean()), 4))
print(dict(zip(feats, rf.feature_importances_.round(3))))

Accuracy: 0.9917
Majority-class baseline: 0.5255
{'heart_rate': np.float64(0.335), 'respiratory_rate': np.float64(0.001), 'body_temperature': np.float64(0.003), 'oxygen_saturation': np.float64(0.003), 'systolic_blood_pressure': np.float64(0.001), 'diastolic_blood_pressure': np.float64(0.001), 'age': np.float64(0.002), 'weight_kg': np.float64(0.351), 'height_m': np.float64(0.303)}


In [9]:
import mlflow, time, json
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("patient-risk-prediction")

models = {
 "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
 "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
 "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1),
 "Neural Network": make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,32), early_stopping=True, max_iter=200, random_state=42)),
}

rows = []
for name, m in models.items():
    with mlflow.start_run(run_name=name):
        t0 = time.time(); m.fit(Xtr, ytr); train_s = time.time() - t0
        t0 = time.time(); pred = m.predict(Xte); inf_ms = (time.time() - t0) / len(Xte) * 1000
        proba = m.predict_proba(Xte)[:, 1]
        acc = accuracy_score(yte, pred)
        p, r, f1, _ = precision_recall_fscore_support(yte, pred, average='macro', zero_division=0)
        auc = roc_auc_score(yte, proba)
        mlflow.log_param("model", name)
        mlflow.log_param("features", ",".join(feats))
        mlflow.log_metrics({"accuracy":acc, "macro_precision":p, "macro_recall":r,
                            "macro_f1":f1, "roc_auc":auc,
                            "train_seconds":train_s, "inference_ms_per_row":inf_ms})
        rows.append([name, acc, p, r, f1, auc, train_s, inf_ms])
        print(f"{name}: acc={acc:.4f} macroF1={f1:.4f} AUC={auc:.4f} train={train_s:.1f}s")

res = pd.DataFrame(rows, columns=["Model","Accuracy","Macro Precision","Macro Recall","Macro F1","ROC AUC","Train s","Inference ms/row"])
res = res.sort_values("Macro F1", ascending=False).round(4)
res.to_csv("results.csv", index=False)
res

Logistic Regression: acc=0.6354 macroF1=0.6338 AUC=0.7191 train=0.1s
Random Forest: acc=0.9917 macroF1=0.9917 AUC=0.9999 train=23.6s
XGBoost: acc=0.9973 macroF1=0.9973 AUC=1.0000 train=2.1s
Neural Network: acc=0.9975 macroF1=0.9975 AUC=1.0000 train=25.4s


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,ROC AUC,Train s,Inference ms/row
3,Neural Network,0.9975,0.9976,0.9974,0.9975,1.0000,25.3832,0.0017
2,XGBoost,0.9973,0.9973,0.9973,0.9973,1.0000,2.0679,0.0033
1,Random Forest,0.9917,0.9914,0.9920,0.9917,0.9999,23.6371,0.0050
0,Logistic Regression,0.6354,0.6342,0.6337,0.6338,0.7191,0.1389,0.0001


In [10]:
mlflow.search_runs(experiment_names=["patient-risk-prediction"])[
    ["tags.mlflow.runName","metrics.accuracy","metrics.macro_f1","metrics.roc_auc","metrics.train_seconds","metrics.inference_ms_per_row"]]

,tags.mlflow.runName,metrics.accuracy,metrics.macro_f1,metrics.roc_auc,metrics.train_seconds,metrics.inference_ms_per_row
0,Neural Network,0.997500,0.997493,0.999983,25.383219,0.001656
1,XGBoost,0.997300,0.997293,0.999970,2.067939,0.003277
2,Random Forest,0.991676,0.991660,0.999905,23.637148,0.005024
3,Logistic Regression,0.635386,0.633816,0.719146,0.138858,0.000133
4,Neural Network,0.997500,0.997493,0.999983,24.189839,0.001318
5,XGBoost,0.997300,0.997293,0.999970,2.634982,0.003287
6,Random Forest,0.991676,0.991660,0.999905,25.099585,0.007129
7,Logistic Regression,0.635386,0.633816,0.719146,0.232689,0.000136


In [12]:
import json, boto3, os
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxruntime as ort
import numpy as np

xgb = models["XGBoost"]

# Fix: reset feature names so the ONNX converter can index them properly
xgb.get_booster().feature_names = None

onx = convert_xgboost(xgb, initial_types=[("input", FloatTensorType([None, len(feats)]))], target_opset=15)
open("model.onnx","wb").write(onx.SerializeToString())
json.dump({"classes":["Low Risk","High Risk"], "features":feats}, open("metadata.json","w"))

s = ort.InferenceSession("model.onnx")
out = s.run(None, {"input": Xte.iloc[:3].values.astype(np.float32)})
print("ONNX labels:", out[0], " XGB labels:", xgb.predict(Xte.iloc[:3].values))
print("Probabilities:", out[1])
print("Model MB:", os.path.getsize("model.onnx")/1e6)

s3 = boto3.client("s3")
for f in ["model.onnx","metadata.json","results.csv"]:
    s3.upload_file(f, "patient-risk-mlops", f"artifacts/{f}")
print("Uploaded to S3 artifacts/")

ONNX labels: [0 1 1]  XGB labels: [0 1 1]
Probabilities: [[9.9889839e-01 1.1016130e-03]
 [1.1920929e-07 9.9999988e-01]
 [4.5418739e-05 9.9995458e-01]]
Model MB: 0.354954
Uploaded to S3 artifacts/


In [2]:
!pip install -q onnxmltools

In [13]:
import boto3

s3 = boto3.client("s3")
for f in ["model.onnx", "metadata.json", "results.csv"]:
    s3.upload_file(f, "patient-risk-mlops", f"artifacts/{f}")

print("Version 2 uploaded successfully to S3!")

Version 2 uploaded successfully to S3!


In [2]:
!rm -rf python onnx_layer.zip
!mkdir -p python
!pip install -q --target python --platform manylinux_2_28_x86_64 --platform manylinux2014_x86_64 --implementation cp --python-version 3.12 --only-binary=:all: onnxruntime numpy
!ls python | head

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sagemaker 2.257.7 requires packaging<25,>=23.0, but you have packaging 26.3 which is incompatible.
sagemaker 2.257.7 requires protobuf<7.0,>=3.12, but you have protobuf 7.36.2 which is incompatible.
bin
flatbuffers
flatbuffers-25.12.19.dist-info
google
numpy
numpy-2.5.4.dist-info
numpy.libs
onnxruntime
onnxruntime-1.31.1.dist-info
packaging


In [3]:
import shutil, boto3, os
shutil.make_archive("onnx_layer", "zip", root_dir=".", base_dir="python")
print("Layer size MB:", round(os.path.getsize("onnx_layer.zip")/1e6, 1))
boto3.client("s3").upload_file("onnx_layer.zip", "patient-risk-mlops", "artifacts/layer/onnx_layer.zip")
print("Uploaded layer")

Layer size MB: 45.4
Uploaded layer
